In [5]:
# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# Imports
import torch
import json
import re
import time
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM

DATA_DIR = "/content/drive/MyDrive/cs639FM"
df = pd.read_csv(f"{DATA_DIR}/aime_historical.csv")
print(f"Total problems: {len(df)}")
df.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Total problems: 933


,ID,Year,Problem Number,Question,Answer,Part
0,1983-1,1983,1,"Let $x$ , $y$ and $z$ all exceed $1$ and let $...",60,NaN
1,1983-2,1983,2,"Let $f(x)=|x-p|+|x-15|+|x-p-15|$ , where $0 < ...",15,NaN
2,1983-3,1983,3,What is the product of the real roots of the e...,20,NaN
3,1983-4,1983,4,A machine-shop cutting tool has the shape of a...,26,NaN
4,1983-5,1983,5,Suppose that the sum of the squares of two com...,4,NaN


In [6]:
##load R1 for trace
model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16,
    device_map="auto"
)
print("DeepSeek loaded!")

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

DeepSeek loaded!


In [7]:
def extract_cot(trace):
    """Extract content inside <think>...</think> tags."""
    match = re.search(r'<think>(.*?)</think>', trace, re.DOTALL)
    if match:
        return match.group(1).strip()
    return trace.strip()

def generate_trace(problem, model, tokenizer, max_new_tokens=4096, temperature=0.7):
    """Generate one reasoning trace."""
    inputs = tokenizer(problem, return_tensors="pt").to(model.device)
    output = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        do_sample=True
    )
    trace = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return trace

def batch_generate(df, model, tokenizer, num_problems=10, traces_per_problem=3,
                   save_path=f"{DATA_DIR}/traces.jsonl"):
    """Generate multiple traces per problem, save incrementally."""

    # Load existing results if resuming
    results = []
    done_ids = set()
    try:
        with open(save_path) as f:
            for line in f:
                r = json.loads(line)
                results.append(r)
                done_ids.add((r["problem_id"], r["trace_id"]))
        print(f"Resuming: {len(results)} traces already done")
    except FileNotFoundError:
        pass

    total = num_problems * traces_per_problem
    count = len(results)

    for idx in range(min(num_problems, len(df))):
        row = df.iloc[idx]
        problem = row['Question']
        answer = str(row['Answer'])
        problem_id = row['ID']

        for t in range(traces_per_problem):
            # Skip if already done
            if (problem_id, t) in done_ids:
                continue

            count += 1
            print(f"[{count}/{total}] {problem_id} trace {t+1}/{traces_per_problem}...", end=" ")
            start = time.time()

            try:
                trace = generate_trace(problem, model, tokenizer)
                cot = extract_cot(trace)

                result = {
                    "problem_id": problem_id,
                    "problem": problem,
                    "ground_truth": answer,
                    "trace_id": t,
                    "full_trace": trace,
                    "cot": cot,
                }
                results.append(result)

                elapsed = time.time() - start
                print(f"Done ({elapsed:.1f}s, {len(cot)} chars)")

                # Save after each trace
                with open(save_path, 'w') as f:
                    for r in results:
                        f.write(json.dumps(r) + "\n")

            except Exception as e:
                print(f"ERROR: {e}")

    print(f"\nDone! {len(results)} total traces saved to {save_path}")
    return results

In [9]:
results = batch_generate(
    df, model, tokenizer,
    num_problems=3,       # start small, increase later
    traces_per_problem=3,
    save_path=f"{DATA_DIR}/traces.jsonl"
)

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


[1/9] 1983-1 trace 1/3... 

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Done (88.9s, 6019 chars)
[2/9] 1983-1 trace 2/3... 

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


...

Done (125.1s, 9719 chars)
[7/9] 1983-3 trace 1/3... 

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Done (110.5s, 6574 chars)
[8/9] 1983-3 trace 2/3... 

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Done (32.9s, 2434 chars)
[9/9] 1983-3 trace 3/3... Done (60.1s, 3808 chars)

Done! 9 total traces saved to /content/drive/MyDrive/cs639FM/traces.jsonl


In [10]:
del model, tokenizer
import gc
gc.collect()
torch.cuda.empty_cache()

model_name = "Qwen/Qwen2.5-7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16,
    device_map="auto"
)
print("Qwen Instruct loaded!")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Qwen Instruct loaded!


In [11]:
def judge_trace_free(cot_text, model, tokenizer, max_new_tokens=4096):
    """Tag a CoT trace with [STRATEGY] and [EXECUTION] labels."""

    prompt = f"""You are an expert at analyzing mathematical reasoning traces.

Read the following reasoning trace and wrap every part of the text with either [STRATEGY]...[/STRATEGY] or [EXECUTION]...[/EXECUTION] tags.

- STRATEGY: choosing, planning, deciding what approach to take, reconsidering
- EXECUTION: computing, substituting, simplifying, algebraic manipulation

Rules:
- Every word in the trace must be inside one of the two tags
- You can split ANYWHERE - mid-sentence, mid-paragraph - wherever the type changes
- Output the EXACT original text, just with tags added around it
- Do not summarize or skip any text

Example:
[STRATEGY]Let me try using the change of base formula.[/STRATEGY][EXECUTION]We have log_x(w) = ln(w)/ln(x) = 24, so ln(w) = 24 ln(x).[/EXECUTION][STRATEGY]Now I need a way to connect x, y, and z. Let me think about what happens with log_xyz(w).[/STRATEGY]

Reasoning trace:
\"\"\"
{cot_text}
\"\"\"

Tagged trace:"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=0.3,
        do_sample=True
    )
    raw = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return raw

def parse_tagged_trace(raw):
    """Parse tagged output into segments, removing repeats."""
    pattern = r'\[(STRATEGY|EXECUTION)\](.*?)\[/\1\]'
    matches = re.findall(pattern, raw, re.DOTALL)

    segments = []
    seen_texts = set()
    for label, text in matches:
        text = text.strip()
        if text in seen_texts:
            break
        seen_texts.add(text)
        segments.append({
            "label": label,
            "text": text
        })
    return segments

def batch_label(traces_path, save_path, model, tokenizer, max_chars=3000):
    """Label all traces and save."""

    with open(traces_path) as f:
        traces = [json.loads(line) for line in f]

    # Load existing if resuming
    labeled = []
    done_keys = set()
    try:
        with open(save_path) as f:
            for line in f:
                r = json.loads(line)
                labeled.append(r)
                done_keys.add((r["problem_id"], r["trace_id"]))
        print(f"Resuming: {len(labeled)} already labeled")
    except FileNotFoundError:
        pass

    for i, t in enumerate(traces):
        key = (t["problem_id"], t["trace_id"])
        if key in done_keys:
            continue

        print(f"[{len(labeled)+1}/{len(traces)}] Labeling {t['problem_id']} trace {t['trace_id']}...", end=" ")
        start = time.time()

        try:
            raw = judge_trace_free(t["cot"][:max_chars], model, tokenizer)
            segments = parse_tagged_trace(raw)

            result = {
                "problem_id": t["problem_id"],
                "problem": t["problem"],
                "ground_truth": t["ground_truth"],
                "trace_id": t["trace_id"],
                "cot": t["cot"],
                "segments": segments
            }
            labeled.append(result)

            elapsed = time.time() - start
            strat = sum(1 for s in segments if s["label"] == "STRATEGY")
            exe = sum(1 for s in segments if s["label"] == "EXECUTION")
            print(f"Done ({elapsed:.1f}s, {len(segments)} segments: {strat}S/{exe}E)")

            # Save after each
            with open(save_path, 'w') as f:
                for r in labeled:
                    f.write(json.dumps(r) + "\n")

        except Exception as e:
            print(f"ERROR: {e}")

    print(f"\nDone! {len(labeled)} labeled traces saved to {save_path}")
    return labeled

In [12]:
labeled = batch_label(
    traces_path=f"{DATA_DIR}/traces.jsonl",
    save_path=f"{DATA_DIR}/labeled_traces.jsonl",
    model=model,
    tokenizer=tokenizer
)

[1/9] Labeling 1983-1 trace 0... Done (57.2s, 53 segments: 6S/47E)
[2/9] Labeling 1983-1 trace 1... Done (144.8s, 2 segments: 1S/1E)
[3/9] Labeling 1983-1 trace 2... Done (144.2s, 42 segments: 29S/13E)
[4/9] Labeling 1983-2 trace 0... Done (50.5s, 47 segments: 24S/23E)
[5/9] Labeling 1983-2 trace 1... Done (56.6s, 5 segments: 5S/0E)
[6/9] Labeling 1983-2 trace 2... Done (39.9s, 11 segments: 6S/5E)
[7/9] Labeling 1983-3 trace 0... Done (145.1s, 31 segments: 15S/16E)
[8/9] Labeling 1983-3 trace 1... Done (35.2s, 32 segments: 7S/25E)
[9/9] Labeling 1983-3 trace 2... Done (45.3s, 26 segments: 13S/13E)

Done! 9 labeled traces saved to /content/drive/MyDrive/cs639FM/labeled_traces.jsonl


In [13]:
with open(f"{DATA_DIR}/labeled_traces.jsonl") as f:
    labeled = [json.loads(line) for line in f]

total_segments = sum(len(r["segments"]) for r in labeled)
total_strategy = sum(1 for r in labeled for s in r["segments"] if s["label"] == "STRATEGY")
total_execution = total_segments - total_strategy

print(f"Labeled traces: {len(labeled)}")
print(f"Total segments: {total_segments}")
print(f"Strategy: {total_strategy} ({total_strategy/total_segments*100:.1f}%)")
print(f"Execution: {total_execution} ({total_execution/total_segments*100:.1f}%)")

Labeled traces: 9
Total segments: 249
Strategy: 106 (42.6%)
Execution: 143 (57.4%)


In [14]:
del model, tokenizer
import gc
gc.collect()
torch.cuda.empty_cache()

In [15]:
import json

with open(f"{DATA_DIR}/labeled_traces.jsonl") as f:
    labeled = [json.loads(line) for line in f]

# Flatten into (text, label) pairs
dataset = []
for trace in labeled:
    for seg in trace["segments"]:
        dataset.append({
            "text": seg["text"],
            "label": 1 if seg["label"] == "STRATEGY" else 0
        })

print(f"Total training samples: {len(dataset)}")
print(f"Strategy: {sum(d['label'] for d in dataset)}")
print(f"Execution: {sum(1 - d['label'] for d in dataset)}")

Total training samples: 249
Strategy: 106
Execution: 143


In [16]:
!pip install datasets evaluate -q

from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer
)
from datasets import Dataset
from sklearn.model_selection import train_test_split
import numpy as np

# Split into train/test
train_data, test_data = train_test_split(dataset, test_size=0.2, random_state=42,
                                          stratify=[d['label'] for d in dataset])

train_ds = Dataset.from_list(train_data)
test_ds = Dataset.from_list(test_data)

print(f"Train: {len(train_ds)}, Test: {len(test_ds)}")

# Load DistilBERT
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# Tokenize
def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=256)

train_ds = train_ds.map(tokenize, batched=True)
test_ds = test_ds.map(tokenize, batched=True)

# Training
training_args = TrainingArguments(
    output_dir=f"{DATA_DIR}/segmenter_model",
    num_train_epochs=10,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_steps=10,
    seed=42,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = (preds == labels).mean()
    # Per-class accuracy
    strat_mask = labels == 1
    exec_mask = labels == 0
    strat_acc = (preds[strat_mask] == 1).mean() if strat_mask.sum() > 0 else 0
    exec_acc = (preds[exec_mask] == 0).mean() if exec_mask.sum() > 0 else 0
    return {
        "accuracy": acc,
        "strategy_recall": strat_acc,
        "execution_recall": exec_acc
    }

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics,
)

trainer.train()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.3 MB/s eta 0:00:00
Train: 199, Test: 50


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=130, training_loss=0.18366634524785555, metrics={'train_runtime': 49.8441, 'train_samples_per_second': 39.924, 'train_steps_per_second': 2.608, 'total_flos': 131805061662720.0, 'train_loss': 0.18366634524785555, 'epoch': 10.0})

In [17]:
results = trainer.evaluate()
print(f"\nTest Accuracy: {results['eval_accuracy']:.3f}")
print(f"Strategy Recall: {results['eval_strategy_recall']:.3f}")
print(f"Execution Recall: {results['eval_execution_recall']:.3f}")


Test Accuracy: 0.880
Strategy Recall: 0.905
Execution Recall: 0.862


In [18]:
trainer.save_model(f"{DATA_DIR}/segmenter_model/best")
tokenizer.save_pretrained(f"{DATA_DIR}/segmenter_model/best")
print("Segmenter saved!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Segmenter saved!


In [ ]:
##########################Segmenter done################